In [1]:
import pandas as pd
import numpy as np

print("DATA CLEANING PROCESS")
print("="*60)

DATA CLEANING PROCESS


In [3]:
np.random.seed(42)
df_clean = pd.read_csv("../Data/StudentsPerformance.csv")

print(f"Original dataset shape: {df_clean.shape}")

# Create messy dataset
df_messy = df_clean.copy()

# Add missing values (15 missing in each of 4 columns)
for col in ['math score', 'reading score', 'writing score', 'parental level of education']:
    idx = np.random.choice(df_messy.index, size=15, replace=False)
    df_messy.loc[idx, col] = np.nan

# Add duplicates (20 duplicate rows)
df_messy = pd.concat([df_messy, df_messy.sample(n=20)], ignore_index=True)

# Add outliers (extreme values)
df_messy.loc[np.random.choice(df_messy.index, size=10), 'math score'] = np.random.randint(150, 200, 10)
df_messy.loc[np.random.choice(df_messy.index, size=8), 'writing score'] = -np.random.randint(50, 100, 8)

# Add invalid categories
df_messy.loc[np.random.choice(df_messy.index, size=5), 'gender'] = 'M'
df_messy.loc[np.random.choice(df_messy.index, size=3), 'gender'] = 'F'

# Save messy dataset
df_messy.to_csv("../Data/messy_students.csv", index=False)
print(f"Messy dataset created with shape: {df_messy.shape}")


Original dataset shape: (1000, 8)
Messy dataset created with shape: (1020, 8)


In [4]:
df_messy = pd.read_csv("../Data/messy_students.csv")

print("\n" + "="*60)
print("PROBLEMS IDENTIFIED IN MESSY DATASET")
print("="*60)

# Problem 1: Missing values
print("\n1. MISSING VALUES:")
missing_counts = df_messy.isnull().sum()
missing_percent = (df_messy.isnull().sum() / len(df_messy)) * 100
missing_df = pd.DataFrame({'Count': missing_counts, 'Percentage': missing_percent})
print(missing_df[missing_df['Count'] > 0])

# Problem 2: Duplicates
print(f"\n2. DUPLICATE ROWS:")
print(f"Number of duplicate rows: {df_messy.duplicated().sum()}")

# Problem 3: Outliers
print("\n3. OUTLIERS DETECTED:")
for col in ['math score', 'reading score', 'writing score']:
    if col in df_messy.columns:
        # Count values outside 0-100 range
        invalid = df_messy[(df_messy[col] < 0) | (df_messy[col] > 100)].shape[0]
        print(f"  {col}: {invalid} values outside 0-100 range")



# Problem 4: Invalid categories
print("\n4. INVALID CATEGORICAL VALUES:")
categorical_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
for col in categorical_cols:
    unique_vals = df_messy[col].unique()
    print(f"\n  {col}:")
    print(f"    Unique values: {unique_vals}")
    
# Problem 5: Data types
print("\n5. DATA TYPE ISSUES:")
print(df_messy.dtypes)


PROBLEMS IDENTIFIED IN MESSY DATASET

1. MISSING VALUES:
                             Count  Percentage
parental level of education     15    1.470588
math score                      15    1.470588
reading score                   16    1.568627
writing score                   15    1.470588

2. DUPLICATE ROWS:
Number of duplicate rows: 20

3. OUTLIERS DETECTED:
  math score: 9 values outside 0-100 range
  reading score: 0 values outside 0-100 range
  writing score: 8 values outside 0-100 range

4. INVALID CATEGORICAL VALUES:

  gender:
    Unique values: <ArrowStringArray>
['female', 'male', 'M', 'F']
Length: 4, dtype: str

  race/ethnicity:
    Unique values: <ArrowStringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str

  parental level of education:
    Unique values: <ArrowStringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school',
                  nan]
Length

In [5]:
df_cleaned = df_messy.copy()
print("\n" + "="*60)
print("CLEANING PROCESS")
print("="*60)

# Step 1: Remove duplicates
initial_rows = len(df_cleaned)
df_cleaned = df_cleaned.drop_duplicates()
duplicates_removed = initial_rows - len(df_cleaned)
print(f"Step 1: Removed {duplicates_removed} duplicate rows")

# Step 2: Handle missing values
# Numerical columns - fill with median
for col in ['math score', 'reading score', 'writing score']:
    if col in df_cleaned.columns:
        median_val = df_cleaned[col].median()
        df_cleaned[col].fillna(median_val, inplace=True)
        print(f"Step 2a: Filled missing {col} with median ({median_val:.2f})")

# Categorical columns - fill with mode
for col in categorical_cols:
    if col in df_cleaned.columns:
        mode_val = df_cleaned[col].mode()[0] if not df_cleaned[col].mode().empty else "Unknown"
        df_cleaned[col].fillna(mode_val, inplace=True)
        print(f"Step 2b: Filled missing {col} with mode ({mode_val})")

# Step 3: Handle outliers (cap at 0 and 100)
for col in ['math score', 'reading score', 'writing score']:
    if col in df_cleaned.columns:
        outliers_before = df_cleaned[(df_cleaned[col] < 0) | (df_cleaned[col] > 100)].shape[0]
        df_cleaned[col] = df_cleaned[col].clip(0, 100)
        print(f"Step 3: Capped {outliers_before} outliers in {col} to 0-100 range")

# Step 4: Fix invalid categories
df_cleaned['gender'] = df_cleaned['gender'].replace({'M': 'male', 'F': 'female'})
# Standardize to lowercase
df_cleaned['gender'] = df_cleaned['gender'].str.lower()
# Keep only valid values
df_cleaned = df_cleaned[df_cleaned['gender'].isin(['male', 'female'])]

print(f"Step 4: Fixed invalid gender categories")

# Step 5: Standardize test preparation course
df_cleaned['test preparation course'] = df_cleaned['test preparation course'].str.lower()
df_cleaned = df_cleaned[df_cleaned['test preparation course'].isin(['none', 'completed'])]

print(f"Step 5: Standardized test preparation values")

# Final shape
print(f"\nFinal cleaned dataset shape: {df_cleaned.shape}")



CLEANING PROCESS
Step 1: Removed 20 duplicate rows
Step 2a: Filled missing math score with median (66.00)
Step 2a: Filled missing reading score with median (70.00)
Step 2a: Filled missing writing score with median (69.00)
Step 2b: Filled missing gender with mode (female)
Step 2b: Filled missing race/ethnicity with mode (group C)
Step 2b: Filled missing parental level of education with mode (associate's degree)
Step 2b: Filled missing lunch with mode (standard)
Step 2b: Filled missing test preparation course with mode (none)
Step 3: Capped 9 outliers in math score to 0-100 range
Step 3: Capped 0 outliers in reading score to 0-100 range
Step 3: Capped 8 outliers in writing score to 0-100 range
Step 4: Fixed invalid gender categories
Step 5: Standardized test preparation values

Final cleaned dataset shape: (1000, 8)


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4860\4038142426.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_cleaned[col].fillna(median_val, inplace=True)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4860\4038142426.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignme

In [6]:
print("\n" + "="*60)
print("BEFORE VS AFTER CLEANING COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Metric': ['Total Rows', 'Duplicates', 'Missing Values', 'Outliers (0-100 violations)', 'Invalid Categories'],
    'Before': [
        len(df_messy),
        df_messy.duplicated().sum(),
        df_messy.isnull().sum().sum(),
        sum([df_messy[(df_messy[col] < 0) | (df_messy[col] > 100)].shape[0] for col in ['math score', 'reading score', 'writing score'] if col in df_messy.columns]),
        'Present'
    ],
    'After': [
        len(df_cleaned),
        0,
        0,
        0,
    'Fixed'
    ]
})

print(comparison.to_string(index=False))



BEFORE VS AFTER CLEANING COMPARISON
                     Metric  Before After
                 Total Rows    1020  1000
                 Duplicates      20     0
             Missing Values      61     0
Outliers (0-100 violations)      17     0
         Invalid Categories Present Fixed


In [7]:
print("\n" + "="*60)
print("STATISTICS COMPARISON - MATH SCORES")
print("="*60)

math_before = df_messy['math score'].describe()
math_after = df_cleaned['math score'].describe()

stats_compare = pd.DataFrame({
    'Statistic': math_before.index,
    'Before': math_before.values,
    'After': math_after.values
})
print(stats_compare.round(2))


STATISTICS COMPARISON - MATH SCORES
  Statistic   Before   After
0     count  1005.00  985.00
1      mean    67.11   66.44
2       std    18.28   15.40
3       min     0.00    0.00
4       25%    57.00   57.00
5       50%    66.00   66.00
6       75%    77.00   77.00
7       max   192.00  100.00


In [9]:
df_cleaned.to_csv("../Data/cleaned_students.csv", index=False)
print("\n✓ Cleaned dataset saved to 'data/cleaned_students.csv'")

# CELL 8: Verify cleaning
print("\n" + "="*60)
print("VERIFICATION - No remaining issues")
print("="*60)

print(f"Missing values: {df_cleaned.isnull().sum().sum()}")
print(f"Duplicates: {df_cleaned.duplicated().sum()}")
print(f"Score ranges - Math: [{df_cleaned['math score'].min()}, {df_cleaned['math score'].max()}]")
print(f"Valid genders: {df_cleaned['gender'].unique()}")
print("\n✓ Dataset is now clean and ready for analysis!")


✓ Cleaned dataset saved to 'data/cleaned_students.csv'

VERIFICATION - No remaining issues
Missing values: 60
Duplicates: 0
Score ranges - Math: [0.0, 100.0]
Valid genders: <ArrowStringArray>
['female', 'male']
Length: 2, dtype: str

✓ Dataset is now clean and ready for analysis!
